<a href="https://colab.research.google.com/github/savvyoliviya/-Google-play-store-growth-analytics/blob/main/Google_play_store_growth_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

In [2]:
from google.colab import files
uploaded = files.upload()  # select google_play_store_dataset.csv when prompted

Saving google_play_store_dataset.csv to google_play_store_dataset.csv


In [3]:
df = pd.read_csv('google_play_store_dataset.csv')
print(df.shape)
df.head()

(10841, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [4]:
# Row 10472 has a shifted-column corruption (Category missing → everything shifts)
df = df.drop(index=10472).reset_index(drop=True)
print(df.shape)  # should be (10840, 13)

(10840, 13)


In [5]:
# Keep the row with the highest review count per App (most recent/complete scrape)
df['Reviews'] = df['Reviews'].astype(str).str.replace(',', '', regex=False)
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

df = df.sort_values('Reviews', ascending=False).drop_duplicates(subset='App', keep='first')
df = df.reset_index(drop=True)
print(df.shape)

(9659, 13)


In [6]:
df['Installs'] = df['Installs'].astype(str).str.replace(r'[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

In [7]:
df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce').fillna(0)

In [8]:
def clean_size(x):
    x = str(x)
    if 'Varies with device' in x:
        return np.nan
    if 'M' in x:
        return float(x.replace('M', ''))
    if 'k' in x:
        return float(x.replace('k', '')) / 1024  # convert to MB
    return np.nan

df['Size_MB'] = df['Size'].apply(clean_size)

In [9]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')
snapshot_date = df['Last Updated'].max()  # treat latest date in data as "today"
df['Days_Since_Update'] = (snapshot_date - df['Last Updated']).dt.days

In [10]:
# Don't fill Rating blindly with mean — that distorts stats/ML later.
# We'll keep NaNs for now and decide per-analysis whether to drop or impute.
df['Type'] = df['Type'].fillna('Free')  # only 1 missing, safe default
df['Content Rating'] = df['Content Rating'].fillna('Unrated')

In [11]:
print(df.dtypes)
print(df.isnull().sum())
df.describe(include='all')

App                          object
Category                     object
Rating                      float64
Reviews                       int64
Size                         object
Installs                      int64
Type                         object
Price                       float64
Content Rating               object
Genres                       object
Last Updated         datetime64[ns]
Current Ver                  object
Android Ver                  object
Size_MB                     float64
Days_Since_Update             int64
dtype: object
App                     0
Category                0
Rating               1463
Reviews                 0
Size                    0
Installs                0
Type                    0
Price                   0
Content Rating          0
Genres                  0
Last Updated            0
Current Ver             8
Android Ver             2
Size_MB              1228
Days_Since_Update       0
dtype: int64


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Days_Since_Update
count,9659,9659,8196.000000,9.659000e+03,9659,9.659000e+03,9659,9659.000000,9659,9659,9659,9651,9657,8431.000000,9659.000000
unique,9659,33,NaN,NaN,461,NaN,2,NaN,6,118,NaN,2819,33,NaN,NaN
top,Dating Tips For Men,FAMILY,NaN,NaN,Varies with device,NaN,Free,NaN,Everyone,Tools,NaN,Varies with device,4.1 and up,NaN,NaN
freq,1,1877,NaN,NaN,1228,NaN,8905,NaN,7903,828,NaN,1055,2205,NaN,NaN
mean,NaN,NaN,4.173267,2.168041e+05,NaN,7.798170e+06,NaN,1.097231,NaN,NaN,2017-10-30 23:45:23.387514368,NaN,NaN,20.398075,281.010146
min,NaN,NaN,1.000000,0.000000e+00,NaN,0.000000e+00,NaN,0.000000,NaN,NaN,2010-05-21 00:00:00,NaN,NaN,0.008301,0.000000
25%,NaN,NaN,4.000000,2.500000e+01,NaN,1.000000e+03,NaN,0.000000,NaN,NaN,2017-08-07 00:00:00,NaN,NaN,4.600000,22.000000
50%,NaN,NaN,4.300000,9.690000e+02,NaN,1.000000e+05,NaN,0.000000,NaN,NaN,2018-05-04 00:00:00,NaN,NaN,12.000000,96.000000
75%,NaN,NaN,4.500000,2.945350e+04,NaN,1.000000e+06,NaN,0.000000,NaN,NaN,2018-07-17 00:00:00,NaN,NaN,28.000000,366.000000
max,NaN,NaN,5.000000,7.815831e+07,NaN,1.000000e+09,NaN,400.000000,NaN,NaN,2018-08-08 00:00:00,NaN,NaN,100.000000,3001.000000


In [12]:
df.to_csv('play_store_cleaned.csv', index=False)
files.download('play_store_cleaned.csv')  # optional, keeps a local copy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>